In [ ]:
import numpy as np
import pandas as pd
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

In [ ]:
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
class Generator(nn.Module):
  def __init__(self):
    super().__init__()

    self.net = nn.Sequential(
        nn.ConvTranspose2d(100, 64, kernel_size=4, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(512),
        nn.ReLU(True),

        nn.ConvTranspose2d(100, 64, kernel_size=4, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(512),
        nn.ReLU(True),

        nn.ConvTranspose2d(100, 64, kernel_size=4, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(512),
        nn.ReLU(True),

        nn.ConvTranspose2d(100, 64, kernel_size=4, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(512),
        nn.ReLU(True),

        nn.ConvTranspose2d(100, 64, kernel_size=4, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(512),
        nn.ReLU(True),

        nn.ConvTranspose2d(100, 64, kernel_size=4, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(512),
        nn.ReLU(True),

        nn.ConvTranspose2d(100, 64, kernel_size=4, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(512),
        nn.ReLU(True),

        nn.ConvTranspose2d(100, 64, kernel_size=4, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(512),
        nn.ReLU(True),

        nn.ConvTranspose2d(100, 64, kernel_size=4, stride=1, padding=0, bias=False),
        nn.Tanh()
    )

  def forward(self, x):
    return self.net(x)

In [ ]:
class Discriminator(nn.Module):
  def __init__(self):
    super().__init__()

    self.net = nn.Sequential(
        nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1, bias=False),
        nn.LeakyReLU(0.2),

        nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
        nn.LeakyReLU(0.2),

        nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
        nn.LeakyReLU(0.2),

        nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
        nn.LeakyReLU(0.2),

        nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0, bias=False)
    )

  def forward(self, x):
    return self.net(x)

In [ ]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)

In [ ]:
genOptimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
disOptimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [ ]:
criteria = nn.BCEWithLogitsLoss()

In [ ]:
epochs = 40

In [ ]:
for epoch in range(epochs):
  for image, label in dataloader:

    images = image.to(device)

    realImgLabel = torch.ones(images.size(0)).to(device)
    fakeImgLabel = torch.zeros(images.size(0)).to(device)

    z = torch.randn(images.size(0), 100, 1, 1).to(device)

    fakeImages = generator(z)

    realOutput = discriminator(images)
    fakeOutput = discriminator(fakeImages.detach())

    realLoss = criteria(realOutput, realImgLabel)
    fakeLoss = criteria(fakeOutput, fakeImgLabel)

    disLoss = realLoss + fakeLoss

    disOptimizer.zero_grad()
    disLoss.backward()
    disOptimizer.step()

    z = torch.randn(images.size(0), 100, 1, 1).to(device)

    fakeImages2 = generator(z)
    fakeOutput2 = discriminator(fakeImages2)

    genLoss = criteria(fakeOutput2, realImgLabel)

    genOptimizer.zero_grad()
    genLoss.backward()
    genOptimizer.step()

